# Random-Episode Meta Pseudo-Supervised SSL on Google Colab (CIFAR-10)

This notebook trains a pseudo-supervised self-supervised model using a meta-loop
where **each episode randomly samples a fixed number of images** from the full
CIFAR-10 training set, rather than using a sequential sliding window.

Key differences from the sequential meta notebook:
- `meta.sampling_mode: random` — each episode draws `episode_size` images at random.
- `meta.num_episodes` — explicitly controls how many episodes are run.
- `train.optimizer_separate` — optionally split backbone and head into separate
  optimizers with independent LR schedulers.

If the GitHub repository is private, create a GitHub fine-grained personal access
token with repository read access and paste it when prompted in the setup cell.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    print('google.colab is only available inside Google Colab; skipping Drive mount.')

In [ ]:
import getpass
import os
import subprocess
from pathlib import Path
from urllib.parse import quote

PUBLIC_REPO_URL = 'https://github.com/moucheng2017/my_simsiam.git'
BRANCH = 'main'
REPO_DIR = Path('/content/my_simsiam')
GITHUB_TOKEN = ""  # Set to a token string here if the repo is private.


def run(cmd, cwd=None):
    print('+', ' '.join(cmd))
    subprocess.run(cmd, cwd=cwd, check=True)


try:
    from google.colab import userdata
    GITHUB_TOKEN = GITHUB_TOKEN or userdata.get('GITHUB_TOKEN')
except Exception:
    pass

repo_url = PUBLIC_REPO_URL
if GITHUB_TOKEN:
    repo_url = PUBLIC_REPO_URL.replace('https://', f'https://oauth2:{quote(GITHUB_TOKEN, safe="")}@')


def sync_repo():
    if GITHUB_TOKEN:
        run(['git', 'remote', 'set-url', 'origin', repo_url], cwd=REPO_DIR)
    else:
        print('No GitHub token found; keeping the existing origin URL for fetch/pull.')
    run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_DIR)
    run(['git', 'checkout', BRANCH], cwd=REPO_DIR)
    run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO_DIR)

os.chdir('/content')

if not REPO_DIR.exists():
    try:
        run(['git', 'clone', '--branch', BRANCH, repo_url, str(REPO_DIR)])
    except subprocess.CalledProcessError as exc:
        if not GITHUB_TOKEN:
            print('\nClone failed without a token. If this repository is private, paste a GitHub token with read access.')
            token = getpass.getpass('GitHub token (leave blank to stop): ').strip()
            if token:
                repo_url = PUBLIC_REPO_URL.replace('https://', f'https://oauth2:{quote(token, safe="")}@')
                run(['git', 'clone', '--branch', BRANCH, repo_url, str(REPO_DIR)])
            else:
                raise RuntimeError('Repository clone was skipped because no token was provided.') from exc
        else:
            raise
else:
    print(f'Repo already exists at {REPO_DIR}; pulling latest changes from {BRANCH}.')
    try:
        sync_repo()
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            'Failed to fetch the latest repo changes. If the repository is private, make sure a GITHUB_TOKEN Colab secret is set or rerun after deleting /content/my_simsiam so the clone step can prompt for a token.'
        ) from exc

if not REPO_DIR.exists():
    raise FileNotFoundError(f'Expected repository at {REPO_DIR}, but it was not created.')

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())
run(['pip', 'install', '-r', 'requirements_colab.txt'], cwd=REPO_DIR)

In [ ]:
# CIFAR-10 is downloaded automatically by torchvision the first time this cell runs.
# The dataset is ~170 MB and will be saved to Google Drive so it only downloads once.
import os
import torchvision

DATA_DIR = '/content/drive/MyDrive/SSL_exps/data'
os.makedirs(DATA_DIR, exist_ok=True)

print('Checking / downloading CIFAR-10 ...')
torchvision.datasets.CIFAR10(DATA_DIR, train=True, download=True)
torchvision.datasets.CIFAR10(DATA_DIR, train=False, download=True)
print('CIFAR-10 is ready.')

## Random-Episode Meta Pseudo-Supervised Training

Each episode randomly samples `episode_size` unique images from CIFAR-10 and trains
them as pseudo-labelled classes.  The total number of episodes is controlled by
`num_episodes`.

Key parameters (pass via `overrides['meta']`, `overrides['train']`, and `overrides['model']`):

| Parameter | Section | Description |
|---|---|---|
| `sampling_mode` | `meta` | `'random'` for random sampling per episode |
| `episode_size` | `meta` | Number of unique images (pseudo-classes) per episode |
| `num_episodes` | `meta` | Total number of episodes to run |
| `dataset_shuffle_seed` | `meta` | Base seed for reproducible per-episode sampling |
| `reset_classifier_head` | `meta` | Re-initialise the linear head at each episode start |
| `global_lr_schedule` | `meta` | Schedule LR over the whole meta-loop (`True`) or reset per episode (`False`) |
| `negatives_ratio` | `train` | Fraction of each batch sampled as unique negative identities; the rest are extra augmented views of those identities |
| `optimizer_separate` | `train` | If `True`, use separate optimizers for backbone and head |
| `cosine_softmax` | `model` | If `True`, use cosine softmax for classification |
| `l2_norm_backbone_features` | `model` | If `True`, L2-normalize backbone features before projection/prediction |

In [ ]:

from colab_utils import meta_train_from_colab

# ── Resume settings ────────────────────────────────────────────────────────────
# Leave all four variables as None / 0 to start a brand-new meta-training run.
# To resume an interrupted run:
#   1. Set RESUME_FROM_EPISODE to the 0-based episode index you want to restart from.
#   2. Set RESUME_CHECKPOINT to the full path of the .pth file saved at the end
#      of the last *completed* episode.
#   3. Set RESUME_MOTHER_LOG_DIR and RESUME_MOTHER_CKPT_DIR to the mother
#      directory paths printed when the original run started.
RESUME_FROM_EPISODE    = 0
RESUME_CHECKPOINT      = None
RESUME_MOTHER_LOG_DIR  = None
RESUME_MOTHER_CKPT_DIR = None
# ───────────────────────────────────────────────────────────────────────────────
# Training settings:
WARMUP_EPOCHS = 10
BATCH_SIZE = 1024
NUM_EPOCHS_EPISODE = 100
NUM_EPISODES = 10
NEGATIVES_RATIO = 0.25  # fraction of each batch sampled as unique negative image 
EPISODE_SIZE = 10000  # number of unique images randomly sampled per episode
NUM_ITERATIONS_PER_SAMPLE = 20
SAMPLES_PER_EPOCH = int(EPISODE_SIZE // NEGATIVES_RATIO)*NUM_ITERATIONS_PER_SAMPLE
NUM_ITERATIONS_PER_EPOCH = SAMPLES_PER_EPOCH // BATCH_SIZE  # Each iteration has 2 images per sample (query + positive)
OPTIMIZER_SEPARATE = False  # Set True to split backbone/head optimizers
RESET_CLASSIFIER_HEAD = False  # Set True to reset the classifier head at the start of each episode
GLOBAL_LR_SCHEDULE = False  # Set True to use a global learning rate schedule across episodes
SAVING_FREQUENCY = 10  # Save a checkpoint every N epochs within each episode (set to None to only save at end)
COSINE_SOFTMAX = True  # Set True to use cosine softmax for classification
L2_NORM_BACKBONE_FEATURES = True  # Set True to L2-normalize backbone features before projection/prediction

exp_name = f'meta-pseudo-cifar10-random-{NUM_ITERATIONS_PER_EPOCH}iter-{BATCH_SIZE}bs-neg{NEGATIVES_RATIO}-{NUM_EPOCHS_EPISODE}epochs-{NUM_EPISODES}episodes-global-lr{GLOBAL_LR_SCHEDULE}-reset-{RESET_CLASSIFIER_HEAD}-cosine-softmax-{COSINE_SOFTMAX}-l2-norm-{L2_NORM_BACKBONE_FEATURES}'

meta_result = meta_train_from_colab(
    config_file='configs/meta_exps/meta_random_config.yaml',
    project_name='SSL_exps',
    use_drive=True,
    device='cuda',
    download=False,  # CIFAR-10 is pre-downloaded by the cell above.
    overrides={
        'name': exp_name,
        'train': {
            'warmup_epochs': WARMUP_EPOCHS,
            'batch_size': BATCH_SIZE,
            'negatives_ratio': NEGATIVES_RATIO,
            'augment_probability': 1.0,
            'num_epochs': NUM_EPOCHS_EPISODE,
            'stop_at_epoch': NUM_EPOCHS_EPISODE,
            'samples_per_epoch': SAMPLES_PER_EPOCH,
            'optimizer_separate': OPTIMIZER_SEPARATE,  # Set True to split backbone/head optimizers
            'saving_frequency': SAVING_FREQUENCY,  # Save checkpoint every N epochs; None = end of episode only
        },
        'model': {
            'cosine_softmax': COSINE_SOFTMAX,  # Set True to use cosine softmax for classification
            'l2_norm_backbone_features': L2_NORM_BACKBONE_FEATURES,  # Set True to L2-normalize backbone features before projection/prediction
        },
        'meta': {
            'sampling_mode': 'random',
            'episode_size': EPISODE_SIZE,   # number of unique images randomly sampled per episode
            'num_episodes': NUM_EPISODES,      # total number of episodes
            'dataset_shuffle_seed': 42,
            'reset_classifier_head': RESET_CLASSIFIER_HEAD,  # Set True to reset the classifier head at the start of each episode
            'global_lr_schedule': GLOBAL_LR_SCHEDULE,  # Set True to use a global learning rate schedule across episodes
        },
    },
    resume_from_episode=RESUME_FROM_EPISODE,
    resume_checkpoint=RESUME_CHECKPOINT,
    resume_mother_log_dir=RESUME_MOTHER_LOG_DIR,
    resume_mother_ckpt_dir=RESUME_MOTHER_CKPT_DIR,
)
meta_result